# Fit Predictor — local training & EDA (Phase 10)

Interactive companion to `scripts/train_match_predictor.py` — **the script stays the reproducible source of truth**; this notebook imports its functions for dataset exploration, training on a local GPU (e.g. RTX 3070), eval plots, and an ONNX sanity check.

Prereqs: `pip install -e ".[train]"` and a CUDA build of torch. No HF/W&B accounts needed for the local loop.

In [ ]:
import os, sys
# Run from the repo root so `scripts` and `src` import cleanly.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1. Explore the dataset
Confirm the column names + label values for whichever public dataset you use — adjust `positives` below to match.

In [ ]:
import collections
from datasets import load_dataset
from scripts.train_match_predictor import DEFAULT_DATASET

ds = load_dataset(DEFAULT_DATASET)
print(ds)
print("columns:", ds["train"].column_names)
label_col = next(c for c in ds["train"].column_names if "label" in c.lower() or "fit" in c.lower())
print("label distribution:", collections.Counter(ds["train"][label_col]))
ds["train"][0]

## 2. Build loaders + model (reuses the script)

In [ ]:
from transformers import AutoTokenizer
from scripts.train_match_predictor import BASE_MODEL, _make_loaders, _build_module

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
positives = {"good fit", "good", "fit", "1", "true"}  # adjust to your dataset's positive labels
train_loader, test_loader = _make_loaders(tokenizer, DEFAULT_DATASET, positives, batch_size=16, smoke=False)
module = _build_module(BASE_MODEL, lr=2e-4)
print("trainable params:", sum(p.numel() for p in module.parameters() if p.requires_grad))

## 3. Train (a few epochs — minutes on a 3070)

In [ ]:
import pytorch_lightning as pl

trainer = pl.Trainer(max_epochs=3, accelerator="auto", devices=1, logger=False)
trainer.fit(module, train_loader, test_loader)
trainer.validate(module, test_loader)

## 4. Eval — AUC, report, confusion matrix

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

device = next(module.parameters()).device
module.eval()
probs, ys = [], []
with torch.no_grad():
    for b in test_loader:
        logit = module(b["r_ids"].to(device), b["r_mask"].to(device), b["j_ids"].to(device), b["j_mask"].to(device))
        probs += torch.sigmoid(logit).cpu().tolist()
        ys += b["label"].tolist()

preds = [p >= 0.5 for p in probs]
print("AUC:", roc_auc_score(ys, probs))
print(classification_report(ys, preds))
print(confusion_matrix(ys, preds))

## 5. Export to ONNX (matches `src/match_predictor.py`'s contract)

In [ ]:
from pathlib import Path
from scripts.train_match_predictor import _export_onnx

# Note: this folds the LoRA weights into the encoder (merge_and_unload), so re-run
# from cell 2 if you want to keep training after exporting.
_export_onnx(module, tokenizer, Path("artifacts/fit-predictor"))

## 6. Verify what the app will serve
Point the predictor at the local artifact and call exactly the code path the API uses.

In [ ]:
from src.utils.config import settings
settings.match_predictor_model = "v1"
settings.match_predictor_path = "artifacts/fit-predictor"

from src import match_predictor
resume = "Python, RAG pipelines, LLM apps, AWS Lambda, FastAPI."
jd = "Hiring an LLM engineer: Python, RAG, AWS, FastAPI."
print("fit:", match_predictor.predict_fit(resume, jd))

## Next steps
To make it stick outside the notebook, set the env in your shell and restart the API:

```powershell
$env:MATCH_PREDICTOR_PATH = "artifacts/fit-predictor"
$env:MATCH_PREDICTOR_MODEL = "v1"
```

The `Model fit` badge on JobDetail then lights up. To share/deploy off this PC, re-run the script with `--hf-repo <you>/resumeagent-fit-v1 --push` and set `MATCH_PREDICTOR_REPO` + `HF_TOKEN` instead.